# Data Processing Pipelines in Practice
Accompanies **Unit 2.3: Data Processing Pipelines in Practice**.

Mathematical algorithms require design matrices $\mathbf{X} \in \mathbb{R}^{n \times p}$ where every entry is a finite real number. 
Raw business tables violate this requirement: numerical measurements are formatted as strings with symbols (`$`, `,`, `%`), missing values are omitted or tokenized arbitrarily, and categorical levels carry no native ordering.

This notebook builds a production-grade, leak-free preprocessing pipeline on the peer-to-peer loan dataset (`loan_data_train.csv`) matching the transformer architecture discussed in Unit 2.3.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Connect to custom transformers in utils/ml_utils.py
sys.path.append(os.path.abspath('../../utils'))
from ml_utils import VarNumericCleaner, VarImputer, VarCategoricalEncoder, DataFramePipeline

data_path = os.path.abspath('../../data/loan_data_train.csv')
df = pd.read_csv(data_path)
df.shape

### Auditing Storage Types vs. Information Types
In pandas, string formatting forces columns into the `object` dtype. 
Applying the **Difference Test**:
- `Interest.Rate`: A difference of $15\% - 10\% = 5\%$ represents a genuine continuous cost of capital. This is true numeric information stored as an object.
- `Debt.To.Income.Ratio`: A continuous leverage ratio stored with a trailing `%` character.

In [ ]:
print("Storage types:")
print(df[['Interest.Rate', 'Debt.To.Income.Ratio', 'Home.Ownership']].dtypes)

print("\nRaw string values:")
df[['Interest.Rate', 'Debt.To.Income.Ratio']].head(3)

### Stripping Non-Numeric Formatting and Coercing to Float

In [ ]:
cleaner = VarNumericCleaner(variables=['Interest.Rate', 'Debt.To.Income.Ratio'])
df_cleaned = cleaner.fit_transform(df)

df_cleaned[['Interest.Rate', 'Debt.To.Income.Ratio']].dtypes

### Median Imputation on Numerical Features
We impute missing entries using the training column medians rather than means to prevent distortion from extreme outliers.

In [ ]:
imputer = VarImputer()
df_imputed = imputer.fit_transform(df_cleaned)

print("Remaining null values in numerical columns:", df_imputed.select_dtypes(include=[np.number]).isnull().sum().sum())

### Low-Frequency Clubbing and Baseline Dummy Encoding
Categories accounting for less than $1\%$ of records are pooled into `_low_freq_cat_` to prevent overfitting.
One category is dropped as baseline to avoid the dummy variable trap (perfect multicollinearity).

In [ ]:
encoder = VarCategoricalEncoder(freq_cutoff=0.01)
df_final = encoder.fit_transform(df_imputed)

print(f"Final preprocessed design matrix shape: {df_final.shape}")
df_final.head(3)